In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Outlook Decision Tree

This notebook mirrors the weather-based decision tree shown in the lecture. Choose a weather scenario and the notebook highlights the path from the root to the prediction.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import ipywidgets as widgets


In [ ]:
POSITIONS = {
    'Outlook': (5.0, 8.8),
    'Sunny': (2.1, 6.9),
    'Overcast': (5.0, 6.9),
    'Rain': (7.9, 6.9),
    'Humidity': (2.1, 5.0),
    'Windy': (7.9, 5.0),
    'High': (1.2, 3.1),
    'Normal': (3.0, 3.1),
    'False': (7.0, 3.1),
    'True': (8.8, 3.1),
    'PlayNoSunny': (1.2, 1.3),
    'PlayYesSunny': (3.0, 1.3),
    'PlayYesOvercast': (5.0, 5.0),
    'PlayYesRain': (7.0, 1.3),
    'PlayNoRain': (8.8, 1.3),
}

LABELS = {
    'Outlook': 'Outlook',
    'Sunny': 'Sunny',
    'Overcast': 'Overcast',
    'Rain': 'Rainy',
    'Humidity': 'Humidity',
    'Windy': 'Windy?',
    'High': 'High',
    'Normal': 'Normal',
    'False': 'False',
    'True': 'True',
    'PlayNoSunny': 'Play = No',
    'PlayYesSunny': 'Play = Yes',
    'PlayYesOvercast': 'Play = Yes',
    'PlayYesRain': 'Play = Yes',
    'PlayNoRain': 'Play = No',
}

EDGES = [
    ('Outlook', 'Sunny'),
    ('Outlook', 'Overcast'),
    ('Outlook', 'Rain'),
    ('Sunny', 'Humidity'),
    ('Humidity', 'High'),
    ('Humidity', 'Normal'),
    ('High', 'PlayNoSunny'),
    ('Normal', 'PlayYesSunny'),
    ('Overcast', 'PlayYesOvercast'),
    ('Rain', 'Windy'),
    ('Windy', 'False'),
    ('Windy', 'True'),
    ('False', 'PlayYesRain'),
    ('True', 'PlayNoRain'),
]

LEAF_COLORS = {
    'PlayYesSunny': '#d1fae5',
    'PlayYesOvercast': '#d1fae5',
    'PlayYesRain': '#d1fae5',
    'PlayNoSunny': '#fee2e2',
    'PlayNoRain': '#fee2e2',
}

def classify(outlook, humidity, windy):
    if outlook == 'Overcast':
        return 'Play = Yes', ['Outlook', 'Overcast', 'PlayYesOvercast']
    if outlook == 'Sunny':
        if humidity == 'High':
            return 'Play = No', ['Outlook', 'Sunny', 'Humidity', 'High', 'PlayNoSunny']
        return 'Play = Yes', ['Outlook', 'Sunny', 'Humidity', 'Normal', 'PlayYesSunny']
    if windy:
        return 'Play = No', ['Outlook', 'Rain', 'Windy', 'True', 'PlayNoRain']
    return 'Play = Yes', ['Outlook', 'Rain', 'Windy', 'False', 'PlayYesRain']

def draw_tree(outlook='Sunny', temperature='Mild', humidity='High', windy=False):
    prediction, path_nodes = classify(outlook, humidity, windy)
    active_edges = set(zip(path_nodes, path_nodes[1:]))
    active_nodes = set(path_nodes)

    fig, ax = plt.subplots(figsize=(10, 6.4))
    ax.set_xlim(0.3, 9.7)
    ax.set_ylim(0.4, 9.6)
    ax.axis('off')

    for parent, child in EDGES:
        x1, y1 = POSITIONS[parent]
        x2, y2 = POSITIONS[child]
        edge_color = '#d97706' if (parent, child) in active_edges else '#94a3b8'
        edge_width = 2.6 if (parent, child) in active_edges else 1.4
        arrow = FancyArrowPatch((x1, y1 - 0.35), (x2, y2 + 0.35), arrowstyle='-', linewidth=edge_width, color=edge_color)
        ax.add_patch(arrow)

    for node, (x, y) in POSITIONS.items():
        is_leaf = node.startswith('Play')
        face = LEAF_COLORS.get(node, '#eff6ff') if is_leaf else '#f8fafc'
        edge = '#d97706' if node in active_nodes else '#334155'
        lw = 2.6 if node in active_nodes else 1.3
        patch = FancyBboxPatch((x - 0.75, y - 0.32), 1.5, 0.64, boxstyle='round,pad=0.16,rounding_size=0.12', facecolor=face, edgecolor=edge, linewidth=lw)
        ax.add_patch(patch)
        ax.text(x, y, LABELS[node], ha='center', va='center', fontsize=11, weight='bold' if node in active_nodes else 'normal', color='#0f172a')

    path_text = ' -> '.join(LABELS[node] for node in path_nodes)
    windy_label = 'True' if windy else 'False'
    ax.text(0.5, 9.25, 'Selected example', fontsize=13, weight='bold', color='#0f172a')
    ax.text(0.5, 8.75, f'Outlook = {outlook}, Temperature = {temperature}, Humidity = {humidity}, Windy = {windy_label}', fontsize=11, color='#334155')
    ax.text(0.5, 8.25, f'Tree path: {path_text}', fontsize=11, color='#334155')
    ax.text(0.5, 7.7, f'Prediction: {prediction}', fontsize=13, weight='bold', color='#0f766e' if prediction.endswith('Yes') else '#b91c1c')
    ax.text(0.5, 7.15, 'This learned tree ignores Temperature for these splits, so that control stays informational.', fontsize=10, color='#64748b')
    plt.show()

widgets.interact(
    draw_tree,
    outlook=widgets.Dropdown(options=['Sunny', 'Overcast', 'Rain'], value='Sunny', description='Outlook'),
    temperature=widgets.Dropdown(options=['Hot', 'Mild', 'Cool'], value='Mild', description='Temp'),
    humidity=widgets.Dropdown(options=['High', 'Normal'], value='High', description='Humidity'),
    windy=widgets.Checkbox(value=False, description='Windy')
);
